# LTheanine Postlab Analysis <font color = 'blue'>Version 1.3</font>

In [ ]:
import numpy as np
from datascience import *
import math as m
from EDS import *
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')
from bestfit import *

## Part 1: Calculating the Relationship

### Step 1: Importing the Calibration Data
Before we can do analyze our teas, we need to develop a way to relate what we measured (LTheanine/Ninhydrin absorption) with what we want (LTheanine concentration).

Fortunately, we the instructors had prepared L-Theanine Standard Solutions (which we already know the concentrations of). Their absorbances are in the table created below.

In [ ]:
standard_data = Table().with_columns('Concentration (M)', [0.025, 0.035, 0.05, 0.075, 0.1], 
                                     'Absorbance', [0.022461549, 0.116568901, 0.410817957, 0.982969911, 1.700570666])
standard_data

Take a look at the output of this following cell: it shows a scatter plot, with the concentration on the x-axis and the absorbance on the y-axis.

In [ ]:
plt.scatter(standard_data[0], standard_data[1])
plt.title('Maximum Absorbance as a function of L-Theanine Concentration')
plt.ylabel('Absorbance')
plt.xlabel('Concentration (molar)')
plt.show()

Does there seem to be a relationship between the two variables? If so, what kind (linear, square, inverse, etc.)? 

**<Replace this line with your answer!>**

### Step 3: Best Fit
Use the same functions you used in the previous labs to get the `slope`, `intercept`, and $r^2$ of the best fit line.

In [ ]:
# we'll make the arrays for you to analyze:

concentrations = standard_data[0]          # this is your xarray

standard_absorbances = standard_data[1]    # this is your yarray

# and get the slope, intercept, and correlation here.

calibration_slope = slope(..., ...)

calibration_intercept = intercept(..., ...)

calibration_r2 = correlation(..., ...)**2

print(f"The best fit line has a slope of {calibration_slope:2f}, an intercept of {calibration_intercept:2f}, and an r² value of {calibration_r2:2f}.")

Now, assuming you've assigned the slope and intercept to the relevant variable names, run this cell below to see your fit line in relation to the data.

In [ ]:
plt.scatter(standard_data[0], standard_data[1], color = 'black', label = 'actual data')
plt.plot(np.arange(0.02, 0.11, 0.01), [calibration_slope * x + calibration_intercept for x in np.arange(0.02, 0.11, 0.01)], color = 'red', label = 'fit line')
plt.legend()
plt.title('Absorbance as a Function of L-Theanine Concentration')
plt.ylabel('Absorbance')
plt.xlabel('Concentration (micromolar)')
plt.show()

## Part 2: Evaluating *your* data

### Step 1: Importing the data
Now we will use the data you all had collected. Each of your groups had analyzed a group of tea samples, each prepared at a particular steep time and temperature. Now, let's go ahead and use the formula you had developed in Part 1 to turn all of your tea samples' absorptions into concentrations. 

Because there's a lot of samples, we'll make a *custom function* that will take in a whole table of spectra and just output the concentrations of each one. 

The code will assume that the data is in this format below. 

|Wavelength1|Spectrum 1|Wavelength2|Spectrum 2|Wavelength3|Spectrum 3|etc|
|--|--|--|--|--|--|--|
|wavelength|abs|wavelength|abs|wavelength|abs|etc|
|wavelength|abs|wavelength|abs|wavelength|abs|etc|
|wavelength|abs|wavelength|abs|wavelength|abs|etc|
|wavelength|abs|wavelength|abs|wavelength|abs|etc|

In [ ]:
# import the entire ltheanine data here.

# we will assume there are two data tables, but you can follow the pattern to load more if needed

tea_data1 = Table.read_table(...)

tea_data2 = Table.read_table(...)

all_tea_tables = [tea_data1, tea_data2]

Before we dive into the nitty-gritty techy details, let's take a moment to gaze upon our wonderful data. It's a beautiful view, isn't it? Like looking at a sunrise.

Aren't you proud of yourselves? I am certainly proud of you.

In [ ]:
for j in all_tea_tables:
    for i in np.arange(j.num_columns/2):
            plt.scatter(j[int(2*i)], j[int(2*i+1)], label=j.labels[int(2*i+1)].split(':')[0])
plt.legend(bbox_to_anchor = (1.2, 0.2))
plt.title('Spectrums of LTheanine Data')
plt.ylabel('Absorbance')
plt.xlabel('Wavelength (nm)')
plt.show()

As you very likely can see, not all wavelengths are absorbed equally by the purple color. At which wavelengths does the absorbance change most dramatically from sample to sample? This wavelength is called th $\lambda_{max}$ (pronounced lambda max).

**<Replace this with your answer!>**

### Step 2: Extracting Data from the Tea Headers

You and your co-lab-orators should have formatted your column headers in this format:

<center><code><b>TeaType</b>-<b>SteepTime</b>-<b>Temperature</b>:Wavelength (or Absorbance)</code></center>
<br>

Use <code>.split</code> and indices (from lab 2) to separate and isolate the TeaType, SteepTime, and Temperature in the <code>tea_header</code> example below.

You'll be using the lines of code you create here in Step 3. The <code>#!</code> symbols you see are for that future step.

In [ ]:
tea_header = 'GT-20-200:Wavelength (nm)'

In [ ]:
# step one, split at the colon (to get rid of the Wavelength portion)
# the [0] index at the end gets the first term (the part containing our data)
# you should get just 'GT-20-200'

tea_description = tea_header.split(...)[0]   #!

tea_description

In [ ]:
# step two, split the tea_description at the dash, to separate each component
# you should get a list: ['GT', '20', '200']

tea_components = ...                 #!

tea_components

In [ ]:
# step three, use the index to select each term in tea_components
# (the float portion converts from a string to a number)

tea_type = ...                #!

tea_steeptime = float(...)    #!

tea_temperature = float(...)  #!

print(f"The tea type is {tea_type}, the steep time is {tea_steeptime:2f}, and the temperature is {tea_temperature:2f}")

### Step 3: Automating the Extraction of Data
Now, let's make a custom function that will extract the data for us.

In [ ]:
def theanine_data_extract(table, tracker = False):
    """For each column, it finds the absorbance reading closest to 570 nm, 
    and processes it alongside the data in the column header into a list format.
    If you would like to see in real time what the function's doing,
    feel free to set tracker = True as a second argument when running this function! :)"""
    
    collected_final_data = [] # this will store all our results.
    
    for i in np.arange(0, table.num_columns, 2):

        # first, we'll look at the header of each of the columns and extract the Tea type, steep time, and temperature.
        
        tea_header = table.labels[i]

        # now search through step 2 and copy here the Five lines that had the #! at the end

        # make sure they are indented to this level.

        ...

        ...

        ...

        ...

        ...

        tea_temperature_C = (tea_temperature - 32) * 5 / 9   # converts from F to C

        # this part will display your parsed results, if you toggle tracker to be True.

        if tracker == True:
            print(f"Tea #{int(i/2)}'s Data has been parsed!")
            print(f"Tea #{int(i/2)}'s Steep Time: {tea_type}")
            print(f"Tea #{int(i/2)}'s Steep Time: {tea_steeptime:2f} min")
            print(f"Tea #{int(i/2)}'s Temperature: {tea_temperature_C:2f} °C")
        else:
            pass

        # second, we will look at the spectrum itself. (the code for this part is already done for you)

        nearest_readings = table.where(i, are.between(569, 571))   # just in case your wavelengths are not exactly 273

        nearest_readings = nearest_readings.with_column('distance from 570', abs(nearest_readings[i]-570))

        nearest_measurement = nearest_readings.sort('distance from 570')[i+1][0]

        if tracker == True:
            print(f"Tea #{int(i/2)}'s Absorbance at 273 nm: {nearest_measurement}")
        else: pass

        # and now, we will package all the relevant data together into a neat list. :)

        final_data = [tea_type, tea_steeptime, tea_temperature_C, nearest_measurement]

        # this puts that list into an external list, which has all data from all spectrums.

        collected_final_data.append(final_data)

    return collected_final_data
        

### Step 4: Compiling all our data
The following cell will create an empty table for you to put your results into.

The custom function is designed to output a list of lists, which can immediately become the argument for the `.with_rows` method.

Use that method to populate the empty table with our data.

In [ ]:
# empty table; do NOT change this
complete_tea_data = Table().with_columns('Type', [], 'Steep Time', [], 'Temperature', [], 'Absorbance at 570 nm', [])
complete_tea_data

In [ ]:
# use the custom function to get the list of lists

tea_rows1 = ... 

# and then use .with_rows to add those lists to the empty table

complete_tea_data = complete_tea_data.with_rows(...)

# do the same for tea_rows2

tea_rows2 = ...

complete_tea_data = complete_tea_data.with_rows(...)

complete_tea_data

### Step 5: Finding the Concentration
Remember, Part 1 saw you making a best fit line out of our calibration data. This equation allows you to get the concentration of L-Theanine in any solution from just its absorbance.

$$absorbance = slope * concentration + intercept$$

Remember, your $slope$ is stored in `calibration_slope`, and your $intercept$ is stored in `calibration_intercept`. 

Make a custom function that will get the $concentration$ from the absorbance. Do not worry about incorporating the dilution factor.

In [ ]:
def get_conc(absorbance):

    ...

    conc = ...

    return conc

Now, this cell below will return a version of the table that includes the calculated concentration column.

In [ ]:
final_concentrations = []
absorbances = complete_tea_data[3]
types = complete_tea_data[0]
for i in np.arange(complete_tea_data.num_rows):
    uncorrected_conc = get_conc(absorbances[i])
    if types[i] == 'BT':
        final_concentrations.append(uncorrected_conc * 6)

    elif types[i] == 'GT':
        final_concentrations.append(uncorrected_conc * 4)
complete_tea_data = complete_tea_data.with_column('Concentration (M)', final_concentrations)
complete_tea_data.to_csv('Final_LTheanine_Dataset.csv')

complete_tea_data


In the left hand margin you'll see a new file pop up! This is the compiled data, exactly as you and I have prepared it.

Now, it's time to analyze our data. Head to [Tea Trends](https://temple.2i2c.cloud/hub/user-redirect/lab/tree/STEMDS/Chemistry/Tea/Tea_Trends.ipynb).